# Pixels to Predictions — Reproduce v2 Best + Ensemble

**Goal:** Reproduce the 0.768 LB score exactly, then build ensemble for higher score.

**Best known config (v2 + native inference):**
- LoRA r=16, q/k/v/o, 4.16M params
- LR=3e-5, 2 epochs, batch=1, full 3109 samples
- Train on full answer phrase, infer with letter log-prob
- Native image resolution at inference

**Current scores:** Zero-shot 0.635 LB → v2 0.758 val / 0.768 LB → Target: >0.80

**GPU:** A100 (40GB)

In [1]:
# ── Cell 1: Mount Drive & Setup ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/kaggle_final_competition")
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"

for d in [CHECKPOINT_DIR, SUBMISSION_DIR]:
    d.mkdir(exist_ok=True)

print("Data dir contents:", sorted(os.listdir(DATA_DIR)))
print("Existing checkpoints:", sorted(os.listdir(CHECKPOINT_DIR)))

Mounted at /content/drive
Data dir contents: ['images', 'sample_submission.csv', 'test.csv', 'train.csv', 'val.csv']
Existing checkpoints: ['lora_r16_lr0.0002_sanity_ep1_best', 'lora_r16_lr0.0002_sanity_ep1_epoch1_batch12', 'lora_r16_lr0.0002_sanity_ep1_epoch1_batch24', 'lora_r16_lr0.0002_sanity_ep1_epoch1_batch36', 'lora_r16_lr0.0002_sanity_ep1_epoch1_batch48', 'lora_r16_lr0.0002_sanity_ep1_epoch1_final', 'lora_r16_lr0.0002_sanity_ep1_training_log.json', 'lora_r16_lr0.0002_sanity_ep1_val_results.json', 'phase0_val_predictions.csv', 'phase0_zero_shot_results.json']


In [2]:
# ── Cell 2: Install Packages ─────────────────────────────────────────────────
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 149.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.7 MB/s eta 0:00:00


In [3]:
# ── Cell 3: Imports & Config ─────────────────────────────────────────────────
import json
import random
import time
import gc
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ═══════════════════════════════════════════════════════════════════════════
# CONFIG — Exact v2 reproduction
# ═══════════════════════════════════════════════════════════════════════════
MODEL_ID        = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS  = "ABCDEFGHIJ"

# Training — exact v2 config
IMG_SIZE_TRAIN  = 224          # 224 for training (matches v2)
LORA_R          = 16
LORA_ALPHA      = 32
LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj"]
LR              = 3e-5
EPOCHS          = 2
BATCH_SIZE_TRAIN = 4          # A100 can handle 4 (v2 used 1)
WEIGHT_DECAY    = 0.01

# Inference
BATCH_SIZE_INFER = 16         # A100 can handle 16
USE_NATIVE_IMG   = True       # Native resolution at inference (key!)

# Ensemble
ENSEMBLE_SEEDS  = [42, 123, 456]  # 3 models with different seeds

# ═══════════════════════════════════════════════════════════════════════════

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [4]:
# ── Cell 4: Load Data ────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 3109 | Val: 1048 | Test: 1008


In [5]:
# ── Cell 5: Prompt Builder + Image Loader (exact v2) ─────────────────────────

def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def build_prompt(row, include_answer=False):
    """
    Exact v2 prompt template.
    """
    choices = row["choices"]
    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint:    prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        letter = CHOICE_LETTERS[ans]
        prompt += f" {letter}. {choices[ans]}"

    return prompt

def load_image_train(row):
    """224x224 for training (matches v2)."""
    path = DATA_DIR / row["image_path"]
    return Image.open(path).convert("RGB").resize((IMG_SIZE_TRAIN, IMG_SIZE_TRAIN), Image.BICUBIC)

def load_image_native(row):
    """Native resolution for inference (key to 0.768 LB)."""
    path = DATA_DIR / row["image_path"]
    return Image.open(path).convert("RGB")

# Test
row = train_df.iloc[0]
print(build_prompt(row, include_answer=True)[-150:])
print(f"\nTrain image size: {load_image_train(row).size}")
print(f"Native image size: {load_image_native(row).size}")

male will carry his tadpoles through the forest
C. the male's tadpoles will become adult frogs

Answer: C. the male's tadpoles will become adult frogs

Train image size: (224, 224)
Native image size: (302, 252)


In [6]:
# ── Cell 6: Load Model with QLoRA ────────────────────────────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

def load_model_for_training(seed=42):
    """Load fresh base model with QLoRA for training."""
    # Set seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,  # bf16 on A100
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        target_modules=LORA_TARGETS,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} ({trainable/1e6:.2f}M) | Total: {total/1e6:.1f}M")
    print(f"Under 5M limit: {'✅' if trainable <= 5_000_000 else '⚠️ OVER LIMIT'}")

    return model, processor

# Load first model
model, processor = load_model_for_training(seed=42)
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Trainable: 4,161,536 (4.16M) | Total: 306.0M
Under 5M limit: ✅
VRAM used: 0.6 GB


In [7]:
# ── Cell 7: Dataset & Collator (exact v2 approach) ───────────────────────────

class VQATrainDataset(Dataset):
    def __init__(self, df, processor, data_dir, img_size=224):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.data_dir = data_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        image = image.resize((self.img_size, self.img_size), Image.BICUBIC)

        prompt_text = build_prompt(row, include_answer=False)
        full_text = build_prompt(row, include_answer=True)

        # Tokenize both to find where answer starts
        full_inputs = self.processor(
            text=[full_text], images=[image],
            return_tensors="pt", truncation=False
        )
        prompt_inputs = self.processor(
            text=[prompt_text], images=[image],
            return_tensors="pt", truncation=False
        )

        prompt_len = prompt_inputs["input_ids"].shape[1]

        return {
            "input_ids": full_inputs["input_ids"].squeeze(0),
            "attention_mask": full_inputs["attention_mask"].squeeze(0),
            "pixel_values": full_inputs["pixel_values"].squeeze(0),
            "prompt_len": prompt_len,
        }


def collate_train(batch):
    """
    Exact v2 collator: left-pad, mask everything except answer tokens.
    """
    max_len = max(x["input_ids"].shape[0] for x in batch)
    pad_id = processor.tokenizer.pad_token_id

    input_ids_list, attention_mask_list, labels_list, pixel_values_list = [], [], [], []

    for x in batch:
        seq_len = x["input_ids"].shape[0]
        pad_len = max_len - seq_len
        prompt_len = x["prompt_len"]

        # Left-pad
        input_ids = torch.cat([
            torch.full((pad_len,), pad_id, dtype=x["input_ids"].dtype),
            x["input_ids"],
        ])
        attention_mask = torch.cat([
            torch.zeros(pad_len, dtype=x["attention_mask"].dtype),
            x["attention_mask"],
        ])

        # Labels: -100 everywhere except answer tokens
        labels = torch.full_like(input_ids, -100)
        answer_start = pad_len + prompt_len
        labels[answer_start:] = input_ids[answer_start:]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)
        pixel_values_list.append(x["pixel_values"])

    return {
        "input_ids": torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
        "labels": torch.stack(labels_list),
        "pixel_values": torch.stack(pixel_values_list),
    }


# Verify collator on 4 samples
print("Testing collator...")
_test_ds = VQATrainDataset(train_df.head(4), processor, DATA_DIR, IMG_SIZE_TRAIN)
_test_batch = collate_train([_test_ds[j] for j in range(4)])
for i in range(4):
    n_loss = (_test_batch["labels"][i] != -100).sum().item()
    total = (_test_batch["attention_mask"][i] == 1).sum().item()
    answer_tokens = _test_batch["labels"][i][_test_batch["labels"][i] != -100]
    decoded = processor.tokenizer.decode(answer_tokens)
    print(f"  Sample {i}: {n_loss} loss tokens out of {total}, answer='{decoded}'")
del _test_ds, _test_batch
print("\n✅ Collator verified.")

Testing collator...
  Sample 0: 11 loss tokens out of 1682, answer=' C. the male's tadpoles will become adult frogs'
  Sample 1: 9 loss tokens out of 1654, answer=' A. the female's offspring will live longer'
  Sample 2: 10 loss tokens out of 1654, answer=' B. the lioness's cubs will survive attacks'
  Sample 3: 8 loss tokens out of 1657, answer=' B. the gull's offspring will survive'

✅ Collator verified.


In [8]:
# ── Cell 8: Training Function ────────────────────────────────────────────────

def train_model(model, processor, train_df, seed, epochs=EPOCHS, lr=LR):
    """
    Train one model with given seed. Returns the trained model.
    Includes progress bars, loss tracking, and checkpointing.
    """
    exp_name = f"v2_seed{seed}_r{LORA_R}_lr{lr}_ep{epochs}"
    print(f"\n{'='*60}")
    print(f"TRAINING: {exp_name}")
    print(f"{'='*60}")

    # Dataset
    train_dataset = VQATrainDataset(train_df, processor, DATA_DIR, IMG_SIZE_TRAIN)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE_TRAIN,
        shuffle=True,
        collate_fn=collate_train,
        num_workers=2,
        pin_memory=True,
    )

    # Optimizer + Scheduler
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * 0.05)

    # Linear warmup + cosine decay
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    print(f"Batches/epoch: {len(train_loader)} | Total steps: {total_steps}")
    print(f"Warmup steps: {warmup_steps}")

    # Training loop
    model.train()
    training_log = []
    start_time = time.time()

    for epoch in range(epochs):
        epoch_losses = []

        pbar = tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True,
        )

        for batch_idx, batch in pbar:
            batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            epoch_losses.append(loss.item())

            # Progress bar update
            avg_loss = np.mean(epoch_losses[-20:])
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "avg": f"{avg_loss:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                "VRAM": f"{torch.cuda.memory_allocated()/1e9:.1f}GB",
            })

            # Log every 50 batches
            if (batch_idx + 1) % 50 == 0:
                training_log.append({
                    "epoch": epoch + 1,
                    "batch": batch_idx + 1,
                    "loss": loss.item(),
                    "avg_loss": avg_loss,
                    "lr": scheduler.get_last_lr()[0],
                })

            # Mid-epoch checkpoint (every 25%)
            checkpoint_interval = max(len(train_loader) // 4, 1)
            if (batch_idx + 1) % checkpoint_interval == 0 and (batch_idx + 1) < len(train_loader):
                mid_path = CHECKPOINT_DIR / f"{exp_name}_ep{epoch+1}_b{batch_idx+1}"
                model.save_pretrained(str(mid_path))
                print(f"\n  📌 Mid-checkpoint: {mid_path.name}")

        # End of epoch
        epoch_avg = np.mean(epoch_losses)
        print(f"\nEpoch {epoch+1} — Avg loss: {epoch_avg:.4f}")

    # Save final checkpoint
    elapsed = time.time() - start_time
    final_path = CHECKPOINT_DIR / f"{exp_name}_final"
    model.save_pretrained(str(final_path))

    # Save training log
    log_path = CHECKPOINT_DIR / f"{exp_name}_log.json"
    with open(log_path, "w") as f:
        json.dump(training_log, f, indent=2)

    print(f"\n⭐ Model saved: {final_path.name}")
    print(f"Training time: {elapsed/60:.1f} min")

    return model, exp_name

print("Training function ready.")

Training function ready.


In [9]:
# ── Cell 9: Inference Function (log-prob + native image) ─────────────────────

def get_candidate_token_ids(tokenizer):
    """Get all possible token IDs for each choice letter.
    Scores multiple forms: 'A', ' A', '\nA', 'A.', ' A.'
    """
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " " + letter, "\n" + letter, letter + ".", " " + letter + "."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc:
                ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

candidate_ids = get_candidate_token_ids(processor.tokenizer)
print("Candidate token IDs:")
for letter in CHOICE_LETTERS[:5]:
    print(f"  {letter}: {candidate_ids[letter]}")


def predict_batch_logprob(model, df_batch, use_native=True):
    """Predict using log-prob scoring at last token position."""
    if use_native:
        images = [load_image_native(row) for _, row in df_batch.iterrows()]
    else:
        images = [load_image_train(row) for _, row in df_batch.iterrows()]

    prompts = [build_prompt(row, include_answer=False) for _, row in df_batch.iterrows()]

    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]  # last position

    log_probs = torch.log_softmax(logits, dim=-1)
    preds = []
    scores_all = []

    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = []
        for ci in range(len(row["choices"])):
            tids = candidate_ids[CHOICE_LETTERS[ci]]
            # Max log-prob across all token forms for this letter
            scores.append(log_probs[i, tids].max().item())
        preds.append(int(np.argmax(scores)))
        scores_all.append(scores)

    return preds, scores_all


def evaluate_full(model, df, use_native=True, batch_size=BATCH_SIZE_INFER, desc="Eval"):
    """Run evaluation on full dataframe with progress bar."""
    model.eval()
    all_preds = []
    all_scores = []
    correct = 0
    total = 0

    pbar = tqdm(range(0, len(df), batch_size), desc=desc, leave=True)

    for start in pbar:
        end = min(start + batch_size, len(df))
        batch_df = df.iloc[start:end]

        preds, scores = predict_batch_logprob(model, batch_df, use_native=use_native)
        all_preds.extend(preds)
        all_scores.extend(scores)

        # Track running accuracy if answers available
        if "answer" in df.columns:
            for p, (_, row) in zip(preds, batch_df.iterrows()):
                total += 1
                if p == int(row["answer"]):
                    correct += 1
            pbar.set_postfix({"acc": f"{correct/total:.4f} ({correct}/{total})"})

        torch.cuda.empty_cache()

    acc = correct / total if total > 0 else None
    return all_preds, all_scores, acc


print("Inference functions ready.")

Candidate token IDs:
  A: [30, 49, 330]
  B: [30, 50, 389]
  C: [30, 51, 340]
  D: [30, 52, 422]
  E: [30, 53, 414]
Inference functions ready.


In [10]:
# ── Cell 10: Sanity Check — Train on 50 samples, verify loss drops ───────────
print("Quick sanity check: 50 samples, 1 epoch...")

sanity_df = train_df.sample(n=50, random_state=42).reset_index(drop=True)
sanity_ds = VQATrainDataset(sanity_df, processor, DATA_DIR, IMG_SIZE_TRAIN)
sanity_loader = DataLoader(
    sanity_ds, batch_size=BATCH_SIZE_TRAIN, shuffle=True,
    collate_fn=collate_train, num_workers=0
)

model.train()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
)

losses = []
for batch in tqdm(sanity_loader, desc="Sanity"):
    batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
    out = model(**batch)
    out.loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    losses.append(out.loss.item())

print(f"\nFirst 3 losses: {losses[:3]}")
print(f"Last 3 losses:  {losses[-3:]}")
print(f"Loss trend: {np.mean(losses[:3]):.4f} → {np.mean(losses[-3:]):.4f}")

if np.mean(losses[-3:]) < np.mean(losses[:3]):
    print("✅ Loss is decreasing — training works!")
else:
    print("⚠️ Loss not clearly decreasing — check config")

# Clean up sanity run
del optimizer, sanity_loader, sanity_ds
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Quick sanity check: 50 samples, 1 epoch...


Sanity:   0%|          | 0/13 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



First 3 losses: [0.17633382976055145, 0.48600101470947266, 0.7584306597709656]
Last 3 losses:  [0.2523157298564911, 0.20897969603538513, 0.05188284441828728]
Loss trend: 0.4736 → 0.1711
✅ Loss is decreasing — training works!
VRAM: 1.7 GB


In [11]:
# ── Cell 11: Full Training — Seed 42 (reproduce v2) ──────────────────────────
# WARNING: This resets the model. If you already ran the sanity check,
# reload the model fresh.

# Reload fresh model
del model
gc.collect()
torch.cuda.empty_cache()

model, processor = load_model_for_training(seed=42)
model, exp_name_42 = train_model(model, processor, train_df, seed=42)

Trainable: 4,161,536 (4.16M) | Total: 306.0M
Under 5M limit: ✅

TRAINING: v2_seed42_r16_lr3e-05_ep2
Batches/epoch: 778 | Total steps: 1556
Warmup steps: 77


Epoch 1/2:   0%|          | 0/778 [00:00<?, ?it/s]


  📌 Mid-checkpoint: v2_seed42_r16_lr3e-05_ep2_ep1_b194

  📌 Mid-checkpoint: v2_seed42_r16_lr3e-05_ep2_ep1_b388

  📌 Mid-checkpoint: v2_seed42_r16_lr3e-05_ep2_ep1_b582

  📌 Mid-checkpoint: v2_seed42_r16_lr3e-05_ep2_ep1_b776

Epoch 1 — Avg loss: 0.1184


Epoch 2/2:   0%|          | 0/778 [00:00<?, ?it/s]


  📌 Mid-checkpoint: v2_seed42_r16_lr3e-05_ep2_ep2_b194

  📌 Mid-checkpoint: v2_seed42_r16_lr3e-05_ep2_ep2_b388

  📌 Mid-checkpoint: v2_seed42_r16_lr3e-05_ep2_ep2_b582

  📌 Mid-checkpoint: v2_seed42_r16_lr3e-05_ep2_ep2_b776

Epoch 2 — Avg loss: 0.0636

⭐ Model saved: v2_seed42_r16_lr3e-05_ep2_final
Training time: 66.7 min


In [15]:
# ── Cell 12: Validate Seed 42 Model ──────────────────────────────────────────
# Clean training memory
gc.collect()
torch.cuda.empty_cache()

print("Evaluating seed 42 model on val set (native images)...")
val_preds_42, val_scores_42, val_acc_42 = evaluate_full(
    model, val_df, use_native=True, desc="Val (seed42)"
)

print(f"\n{'='*60}")
print(f"SEED 42 RESULTS")
print(f"{'='*60}")
print(f"Val Accuracy:  {val_acc_42:.4f}")
print(f"Target (v2):   0.670")
print(f"TA Baseline:   0.678")
print(f"{'='*60}")

# Save
results_42 = {"seed": 42, "val_acc": val_acc_42, "exp": exp_name_42}
with open(CHECKPOINT_DIR / f"{exp_name_42}_val_results.json", "w") as f:
    json.dump(results_42, f, indent=2)
print(f"Results saved.")

Evaluating seed 42 model on val set (native images)...


Val (seed42):   0%|          | 0/66 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [13]:
# ── Debug: Check what token 30 is and verify scoring ─────────────────────────
print("Token 30 decodes to:", repr(processor.tokenizer.decode([30])))
print()

# Check what each candidate ID actually decodes to
for letter in CHOICE_LETTERS[:5]:
    tids = candidate_ids[letter]
    for tid in tids:
        print(f"  {letter}: token_id={tid} → {repr(processor.tokenizer.decode([tid]))}")
    print()

# Manual test: run one sample and check raw logits
sample = val_df.iloc[0]
img = load_image_native(sample)
prompt = build_prompt(sample, include_answer=False)

inputs = processor(text=[prompt], images=[img], return_tensors="pt")
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    logits = model(**inputs).logits

# Check logits at last position
last_logits = logits[0, -1, :]
log_probs = torch.log_softmax(last_logits, dim=-1)

print("Log-probs for each choice letter (all forms):")
for letter in CHOICE_LETTERS[:sample["num_choices"]]:
    tids = candidate_ids[letter]
    for tid in tids:
        print(f"  {letter} (tid={tid}, '{repr(processor.tokenizer.decode([tid]))}'): {log_probs[tid].item():.4f}")

# What does the model actually want to generate?
top_tokens = torch.topk(log_probs, k=10)
print("\nTop 10 predicted tokens:")
for score, tid in zip(top_tokens.values, top_tokens.indices):
    print(f"  {repr(processor.tokenizer.decode([tid.item()]))} (id={tid.item()}): {score.item():.4f}")

print(f"\nGround truth: {CHOICE_LETTERS[int(sample['answer'])]}")

Token 30 decodes to: '.'

  A: token_id=30 → '.'
  A: token_id=49 → 'A'
  A: token_id=330 → ' A'

  B: token_id=30 → '.'
  B: token_id=50 → 'B'
  B: token_id=389 → ' B'

  C: token_id=30 → '.'
  C: token_id=51 → 'C'
  C: token_id=340 → ' C'

  D: token_id=30 → '.'
  D: token_id=52 → 'D'
  D: token_id=422 → ' D'

  E: token_id=30 → '.'
  E: token_id=53 → 'E'
  E: token_id=414 → ' E'

Log-probs for each choice letter (all forms):
  A (tid=30, ''.''): -13.6260
  A (tid=49, ''A''): -9.7909
  A (tid=330, '' A''): -1.5260
  B (tid=30, ''.''): -13.6260
  B (tid=50, ''B''): -9.0765
  B (tid=389, '' B''): -0.4865
  C (tid=30, ''.''): -13.6260
  C (tid=51, ''C''): -10.5553
  C (tid=340, '' C''): -1.8155

Top 10 predicted tokens:
  ' B' (id=389): -0.4865
  ' A' (id=330): -1.5260
  ' C' (id=340): -1.8155
  ' (' (id=365): -5.9003
  ' b' (id=278): -7.5815
  ' c' (id=265): -7.7154
  ' D' (id=422): -8.6689
  ' a' (id=253): -8.7891
  ' ' (id=216): -8.8438
  ' Answer' (id=19842): -8.9972

Ground truth: 

In [14]:
# ── Fix candidate token IDs ──────────────────────────────────────────────────
def get_candidate_token_ids_fixed(tokenizer):
    """Only keep tokens that are unique to each letter."""
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " " + letter]  # only "A" and " A", no period forms
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc:
                ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

candidate_ids = get_candidate_token_ids_fixed(processor.tokenizer)
print("Fixed candidate token IDs:")
for letter in CHOICE_LETTERS[:5]:
    tids = candidate_ids[letter]
    for tid in tids:
        print(f"  {letter}: token_id={tid} → {repr(processor.tokenizer.decode([tid]))}")

Fixed candidate token IDs:
  A: token_id=49 → 'A'
  A: token_id=330 → ' A'
  B: token_id=50 → 'B'
  B: token_id=389 → ' B'
  C: token_id=51 → 'C'
  C: token_id=340 → ' C'
  D: token_id=52 → 'D'
  D: token_id=422 → ' D'
  E: token_id=53 → 'E'
  E: token_id=414 → ' E'


In [16]:
# ── Debug: Is the model in eval mode? Check raw generation ──────────────────
model.eval()

sample = val_df.iloc[0]
img = load_image_native(sample)
prompt = build_prompt(sample, include_answer=False)

inputs = processor(text=[prompt], images=[img], return_tensors="pt")
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

# Try generation — what does the model actually output?
with torch.inference_mode():
    gen_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)

input_len = inputs["input_ids"].shape[1]
generated = processor.tokenizer.decode(gen_ids[0, input_len:], skip_special_tokens=True)
print(f"Generated: '{generated}'")
print(f"Ground truth: {CHOICE_LETTERS[int(sample['answer'])]}")

# Check: are we looking at the right position?
with torch.inference_mode():
    out = model(**inputs)

print(f"\nLogits shape: {out.logits.shape}")
print(f"Input length: {input_len}")

# Check MULTIPLE positions near the end
for pos in [-1, -2, -3]:
    logits_pos = out.logits[0, pos, :]
    lp = torch.log_softmax(logits_pos, dim=-1)
    top5 = torch.topk(lp, k=5)
    print(f"\nPosition {pos}:")
    for score, tid in zip(top5.values, top5.indices):
        print(f"  {repr(processor.tokenizer.decode([tid.item()]))} : {score.item():.4f}")

Generated: ' B. the leech will not eat for up to a week.

Explanation: The passage'
Ground truth: A

Logits shape: torch.Size([1, 1357, 49280])
Input length: 1357

Position -1:
  ' B' : -0.4865
  ' A' : -1.5260
  ' C' : -1.8155
  ' (' : -5.9003
  ' b' : -7.5815

Position -2:
  ' A' : -0.6725
  ' option' : -1.9738
  ':' : -1.9989
  ' B' : -2.0999
  ' choice' : -2.9696

Position -3:
  'Answer' : -0.0204
  'C' : -4.6893
  'B' : -4.8625
  'Correct' : -6.6510
  'D' : -7.8958


In [17]:
# ── Check: Did the sanity check corrupt the model? ──────────────────────────
# The sanity check in Cell 10 trained on 50 samples BEFORE Cell 11 reloaded
# But Cell 11 should have reloaded fresh. Let's verify.

# Check how many checkpoints exist
import os
print("Checkpoints:")
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    if 'v2_seed42' in f:
        print(f"  {f}")

# Check: run on 10 TRAINING samples — if model memorized, these should be ~100%
train_sample = train_df.head(10).copy()
train_preds, _, train_acc = evaluate_full(
    model, train_sample, use_native=True, batch_size=4, desc="Train check"
)
print(f"\nTrain accuracy (10 samples): {train_acc:.4f}")
print("(Should be ~1.0 if model trained properly)")

# Also check: what was the prompt used during training vs inference?
row = train_df.iloc[0]
print(f"\nTraining prompt ends with: ...{build_prompt(row, include_answer=True)[-80:]}")
print(f"Inference prompt ends with: ...{build_prompt(row, include_answer=False)[-80:]}")

Checkpoints:
  v2_seed42_r16_lr3e-05_ep2_ep1_b194
  v2_seed42_r16_lr3e-05_ep2_ep1_b388
  v2_seed42_r16_lr3e-05_ep2_ep1_b582
  v2_seed42_r16_lr3e-05_ep2_ep1_b776
  v2_seed42_r16_lr3e-05_ep2_ep2_b194
  v2_seed42_r16_lr3e-05_ep2_ep2_b388
  v2_seed42_r16_lr3e-05_ep2_ep2_b582
  v2_seed42_r16_lr3e-05_ep2_ep2_b776
  v2_seed42_r16_lr3e-05_ep2_final
  v2_seed42_r16_lr3e-05_ep2_log.json
  v2_seed42_r16_lr3e-05_ep2_val_results.json


Train check:   0%|          | 0/3 [00:00<?, ?it/s]


Train accuracy (10 samples): 0.3000
(Should be ~1.0 if model trained properly)

Training prompt ends with: ... will become adult frogs

Answer: C. the male's tadpoles will become adult frogs
Inference prompt ends with: ...poles through the forest
C. the male's tadpoles will become adult frogs

Answer:


In [19]:
# Quick fix
print(f"Active adapters: {model.active_adapters}")
print(f"PEFT config: {model.peft_config}")

# Now load checkpoint explicitly and test
from peft import PeftModel
from transformers import AutoModelForVision2Seq, BitsAndBytesConfig

del model
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

ckpt_path = str(CHECKPOINT_DIR / "v2_seed42_r16_lr3e-05_ep2_final")
model = PeftModel.from_pretrained(base_model, ckpt_path)
model.eval()
print(f"Loaded from: {ckpt_path}")

# Test on 10 training samples
train_preds, _, train_acc = evaluate_full(
    model, train_df.head(10), use_native=True, batch_size=4, desc="Train (reloaded)"
)
print(f"\nReloaded train accuracy: {train_acc:.4f}")

Active adapters: ['default']
PEFT config: {'default': LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path='HuggingFaceTB/SmolVLM-500M-Instruct', revision=None, inference_mode=False, r=16, target_modules={'o_proj', 'q_proj', 'k_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)}
Loaded from: /co

Train (reloaded):   0%|          | 0/3 [00:00<?, ?it/s]


Reloaded train accuracy: 0.3000


In [22]:
# ── Debug: Single sample vs batch scoring ────────────────────────────────────
# Test 5 samples ONE AT A TIME (not batched)

model.eval()
for idx in range(5):
    row = train_df.iloc[idx]
    img = load_image_native(row)
    prompt = build_prompt(row, include_answer=False)

    inputs = processor(text=[prompt], images=[img], return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits

    # Last position
    last_lp = torch.log_softmax(logits[0, -1, :], dim=-1)

    n_choices = len(row["choices"])
    scores = []
    for ci in range(n_choices):
        letter = CHOICE_LETTERS[ci]
        tids = candidate_ids[letter]
        s = max(last_lp[tid].item() for tid in tids)
        scores.append(s)

    pred = int(np.argmax(scores))
    actual = int(row["answer"])

    print(f"Sample {idx}: pred={CHOICE_LETTERS[pred]} actual={CHOICE_LETTERS[actual]} {'✓' if pred==actual else '✗'}  scores={[f'{s:.3f}' for s in scores]}")

Sample 0: pred=C actual=C ✓  scores=['-1.734', '-2.172', '-0.344']
Sample 1: pred=B actual=A ✗  scores=['-2.926', '-0.067', '-4.551']
Sample 2: pred=A actual=B ✗  scores=['-0.003', '-5.926', '-8.984']
Sample 3: pred=B actual=B ✓  scores=['-8.492', '-0.006', '-5.254']
Sample 4: pred=B actual=B ✓  scores=['-8.281', '-0.013', '-4.402']


In [23]:
# ── Debug: Check if batched inference has padding issue ──────────────────────

# Take the same 5 samples, run batched
batch_df = train_df.head(5)
images = [load_image_native(row) for _, row in batch_df.iterrows()]
prompts = [build_prompt(row, include_answer=False) for _, row in batch_df.iterrows()]

inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

print(f"Batch input_ids shape: {inputs['input_ids'].shape}")
print(f"Attention mask sums (seq lengths): {inputs['attention_mask'].sum(dim=1).tolist()}")

# Check: is position -1 padding?
pad_id = processor.tokenizer.pad_token_id
for i in range(5):
    last_token = inputs["input_ids"][i, -1].item()
    seq_len = inputs["attention_mask"][i].sum().item()
    real_last = inputs["input_ids"][i, int(seq_len)-1].item()
    print(f"  Sample {i}: position -1 token={last_token} ({'PAD' if last_token == pad_id else 'REAL'}), "
          f"seq_len={int(seq_len)}, real last token={real_last} "
          f"({repr(processor.tokenizer.decode([real_last]))})")

Batch input_ids shape: torch.Size([5, 1671])
Attention mask sums (seq lengths): [1671, 1645, 1379, 1649, 1381]
  Sample 0: position -1 token=42 (REAL), seq_len=1671, real last token=42 (':')
  Sample 1: position -1 token=2 (PAD), seq_len=1645, real last token=42 (':')
  Sample 2: position -1 token=2 (PAD), seq_len=1379, real last token=42 (':')
  Sample 3: position -1 token=2 (PAD), seq_len=1649, real last token=42 (':')
  Sample 4: position -1 token=2 (PAD), seq_len=1381, real last token=42 (':')


In [24]:
# ── FIXED predict function ───────────────────────────────────────────────────

def predict_batch_logprob(model, df_batch, use_native=True):
    """Predict using log-prob scoring at correct last token position."""
    if use_native:
        images = [load_image_native(row) for _, row in df_batch.iterrows()]
    else:
        images = [load_image_train(row) for _, row in df_batch.iterrows()]

    prompts = [build_prompt(row, include_answer=False) for _, row in df_batch.iterrows()]

    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits  # [batch, seq_len, vocab]

    # Find the REAL last position for each sample (not padding)
    seq_lengths = inputs["attention_mask"].sum(dim=1)  # [batch]

    preds = []
    scores_all = []

    for i, (_, row) in enumerate(df_batch.iterrows()):
        last_pos = int(seq_lengths[i].item()) - 1  # last non-pad position
        last_logits = logits[i, last_pos, :]
        log_probs = torch.log_softmax(last_logits, dim=-1)

        scores = []
        for ci in range(len(row["choices"])):
            tids = candidate_ids[CHOICE_LETTERS[ci]]
            scores.append(max(log_probs[tid].item() for tid in tids))
        preds.append(int(np.argmax(scores)))
        scores_all.append(scores)

    return preds, scores_all

# Verify fix on same 5 samples
batch_df = train_df.head(5)
preds, scores = predict_batch_logprob(model, batch_df, use_native=True)
for i, (_, row) in enumerate(batch_df.iterrows()):
    actual = int(row["answer"])
    correct = "YES" if preds[i] == actual else "NO"
    print(f"Sample {i}: pred={CHOICE_LETTERS[preds[i]]} actual={CHOICE_LETTERS[actual]} {correct}  scores={[round(s,3) for s in scores[i]]}")

Sample 0: pred=C actual=C YES  scores=[-1.762, -2.152, -0.34]
Sample 1: pred=B actual=A NO  scores=[-3.045, -0.061, -4.469]
Sample 2: pred=A actual=B NO  scores=[-0.004, -5.926, -8.922]
Sample 3: pred=B actual=B YES  scores=[-8.555, -0.006, -5.223]
Sample 4: pred=B actual=B YES  scores=[-8.391, -0.015, -4.25]


In [25]:
gc.collect()
torch.cuda.empty_cache()

val_preds_42, val_scores_42, val_acc_42 = evaluate_full(
    model, val_df, use_native=True, batch_size=16, desc="Val (seed42 FIXED)"
)

print(f"\n{'='*60}")
print(f"Val Accuracy: {val_acc_42:.4f}")
print(f"Target (v2):  0.670")
print(f"{'='*60}")

Val (seed42 FIXED):   0%|          | 0/66 [00:00<?, ?it/s]


Val Accuracy: 0.6870
Target (v2):  0.670


In [26]:
# ── Cell 13: Test Submission for Seed 42 ─────────────────────────────────────
print("Running test inference (seed 42, native images)...")
test_preds_42, test_scores_42, _ = evaluate_full(
    model, test_df, use_native=True, batch_size=16, desc="Test (seed42)"
)

sub_42 = pd.DataFrame({"id": test_df["id"], "answer": test_preds_42})
sub_42.to_csv(SUBMISSION_DIR / "submission_seed42.csv", index=False)
sub_42.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert len(sub_42) == len(sample_sub), "Length mismatch!"

print(f"\nSubmission saved ({len(sub_42)} rows)")
print(f"Prediction distribution:\n{sub_42['answer'].value_counts().sort_index()}")
print(f"\nUpload submission_seed42.csv to Kaggle!")

from google.colab import files
files.download(str(SUBMISSION_DIR / "submission_seed42.csv"))

Running test inference (seed 42, native images)...


Test (seed42):   0%|          | 0/63 [00:00<?, ?it/s]


Submission saved (1008 rows)
Prediction distribution:
answer
0    361
1    339
2    223
3     81
4      4
Name: count, dtype: int64

Upload submission_seed42.csv to Kaggle!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Ensemble: Train 2 more models with different seeds

**Only proceed if seed 42 gives ~0.758+ val accuracy.**

We train 2 more models (seed 123, 456) and do majority voting.

In [ ]:
# ── Cell 14: Train Seed 123 ──────────────────────────────────────────────────
# Save seed 42 test predictions for ensemble
ensemble_val_preds = [val_preds_42]
ensemble_test_preds = [test_preds_42]
ensemble_val_scores = [val_scores_42]
ensemble_test_scores = [test_scores_42]
ensemble_val_accs = [val_acc_42]

# Free seed 42 model
del model
gc.collect()
torch.cuda.empty_cache()

# Train seed 123
model, processor = load_model_for_training(seed=123)
model, exp_name_123 = train_model(model, processor, train_df, seed=123)

In [ ]:
# ── Cell 15: Validate & Test Seed 123 ────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

val_preds_123, val_scores_123, val_acc_123 = evaluate_full(
    model, val_df, use_native=True, desc="Val (seed123)"
)
test_preds_123, test_scores_123, _ = evaluate_full(
    model, test_df, use_native=True, desc="Test (seed123)"
)

ensemble_val_preds.append(val_preds_123)
ensemble_test_preds.append(test_preds_123)
ensemble_val_scores.append(val_scores_123)
ensemble_test_scores.append(test_scores_123)
ensemble_val_accs.append(val_acc_123)

print(f"\nSeed 123 Val Accuracy: {val_acc_123:.4f}")
print(f"Seed 42 Val Accuracy:  {ensemble_val_accs[0]:.4f}")

# Save
sub_123 = pd.DataFrame({"id": test_df["id"], "answer": test_preds_123})
sub_123.to_csv(SUBMISSION_DIR / f"submission_seed123.csv", index=False)

results_123 = {"seed": 123, "val_acc": val_acc_123, "exp": exp_name_123}
with open(CHECKPOINT_DIR / f"{exp_name_123}_val_results.json", "w") as f:
    json.dump(results_123, f, indent=2)

In [ ]:
# ── Cell 16: Train Seed 456 ──────────────────────────────────────────────────
del model
gc.collect()
torch.cuda.empty_cache()

model, processor = load_model_for_training(seed=456)
model, exp_name_456 = train_model(model, processor, train_df, seed=456)

In [ ]:
# ── Cell 17: Validate & Test Seed 456 ────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

val_preds_456, val_scores_456, val_acc_456 = evaluate_full(
    model, val_df, use_native=True, desc="Val (seed456)"
)
test_preds_456, test_scores_456, _ = evaluate_full(
    model, test_df, use_native=True, desc="Test (seed456)"
)

ensemble_val_preds.append(val_preds_456)
ensemble_test_preds.append(test_preds_456)
ensemble_val_scores.append(val_scores_456)
ensemble_test_scores.append(test_scores_456)
ensemble_val_accs.append(val_acc_456)

print(f"\nSeed 456 Val Accuracy: {val_acc_456:.4f}")
print(f"Seed 123 Val Accuracy: {ensemble_val_accs[1]:.4f}")
print(f"Seed 42 Val Accuracy:  {ensemble_val_accs[0]:.4f}")

sub_456 = pd.DataFrame({"id": test_df["id"], "answer": test_preds_456})
sub_456.to_csv(SUBMISSION_DIR / f"submission_seed456.csv", index=False)

results_456 = {"seed": 456, "val_acc": val_acc_456, "exp": exp_name_456}
with open(CHECKPOINT_DIR / f"{exp_name_456}_val_results.json", "w") as f:
    json.dump(results_456, f, indent=2)

In [ ]:
# ── Cell 18: Ensemble — Majority Vote + Score Average ────────────────────────
from collections import Counter

def majority_vote(pred_lists):
    """Simple majority vote across models."""
    n_samples = len(pred_lists[0])
    ensemble_preds = []
    for i in range(n_samples):
        votes = [preds[i] for preds in pred_lists]
        winner = Counter(votes).most_common(1)[0][0]
        ensemble_preds.append(winner)
    return ensemble_preds

def score_average(score_lists, num_choices_list):
    """Average log-prob scores across models, pick argmax."""
    n_samples = len(score_lists[0])
    ensemble_preds = []
    for i in range(n_samples):
        n_choices = num_choices_list[i]
        avg_scores = []
        for ci in range(n_choices):
            avg = np.mean([scores[i][ci] for scores in score_lists])
            avg_scores.append(avg)
        ensemble_preds.append(int(np.argmax(avg_scores)))
    return ensemble_preds

# ── Validation Ensemble ──
val_num_choices = [len(row["choices"]) for _, row in val_df.iterrows()]
val_actuals = [int(row["answer"]) for _, row in val_df.iterrows()]

# Majority vote
val_majority = majority_vote(ensemble_val_preds)
val_majority_acc = sum(p == a for p, a in zip(val_majority, val_actuals)) / len(val_actuals)

# Score average
val_avg = score_average(ensemble_val_scores, val_num_choices)
val_avg_acc = sum(p == a for p, a in zip(val_avg, val_actuals)) / len(val_actuals)

print(f"{'='*60}")
print(f"ENSEMBLE RESULTS (Validation)")
print(f"{'='*60}")
print(f"Individual models:")
for i, acc in enumerate(ensemble_val_accs):
    print(f"  Seed {ENSEMBLE_SEEDS[i]}: {acc:.4f}")
print(f"\nEnsemble (majority vote): {val_majority_acc:.4f}")
print(f"Ensemble (score average): {val_avg_acc:.4f}")
print(f"\nBest individual:          {max(ensemble_val_accs):.4f}")
print(f"Best ensemble:            {max(val_majority_acc, val_avg_acc):.4f}")
print(f"Improvement:              {max(val_majority_acc, val_avg_acc) - max(ensemble_val_accs):+.4f}")

# ── Test Ensemble ──
test_num_choices = [len(row["choices"]) for _, row in test_df.iterrows()]

test_majority = majority_vote(ensemble_test_preds)
test_avg = score_average(ensemble_test_scores, test_num_choices)

# Use whichever ensemble method worked better on val
if val_avg_acc >= val_majority_acc:
    best_method = "score_average"
    best_test_preds = test_avg
else:
    best_method = "majority_vote"
    best_test_preds = test_majority

print(f"\nUsing {best_method} for test submission.")

In [ ]:
# ── Cell 19: Final Ensemble Submission ───────────────────────────────────────

# Save ensemble submission
sub_ensemble = pd.DataFrame({"id": test_df["id"], "answer": best_test_preds})
sub_ensemble.to_csv(SUBMISSION_DIR / "submission_ensemble.csv", index=False)
sub_ensemble.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

# Also save both ensemble methods
sub_majority = pd.DataFrame({"id": test_df["id"], "answer": test_majority})
sub_majority.to_csv(SUBMISSION_DIR / "submission_majority_vote.csv", index=False)

sub_avg = pd.DataFrame({"id": test_df["id"], "answer": test_avg})
sub_avg.to_csv(SUBMISSION_DIR / "submission_score_avg.csv", index=False)

# Verify
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert len(sub_ensemble) == len(sample_sub), "Length mismatch!"

print(f"{'='*60}")
print(f"SUBMISSIONS SAVED")
print(f"{'='*60}")
print(f"1. submission_seed42.csv      (single model)")
print(f"2. submission_seed123.csv     (single model)")
print(f"3. submission_seed456.csv     (single model)")
print(f"4. submission_majority_vote.csv (ensemble)")
print(f"5. submission_score_avg.csv   (ensemble)")
print(f"6. submission.csv             (best: {best_method})")
print(f"\nPrediction distribution (ensemble):")
print(sub_ensemble['answer'].value_counts().sort_index())
print(f"\n→ Upload submission.csv to Kaggle!")

# Save full ensemble results
ensemble_results = {
    "individual_val_accs": {str(s): a for s, a in zip(ENSEMBLE_SEEDS, ensemble_val_accs)},
    "majority_vote_val_acc": val_majority_acc,
    "score_average_val_acc": val_avg_acc,
    "best_method": best_method,
    "config": {
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lr": LR,
        "epochs": EPOCHS, "seeds": ENSEMBLE_SEEDS,
        "targets": LORA_TARGETS, "native_inference": USE_NATIVE_IMG,
    }
}
with open(CHECKPOINT_DIR / "ensemble_results.json", "w") as f:
    json.dump(ensemble_results, f, indent=2)
print(f"\nEnsemble results saved to checkpoints.")

## Run Order

1. **Cells 1-9:** Setup, data, model, functions
2. **Cell 10:** Sanity check (50 samples) — verify loss decreases
3. **Cell 11:** Full training seed 42 — reproduce v2
4. **Cell 12:** Validate seed 42 — should get ~0.67-0.68 val
5. **Cell 13:** Test + submission seed 42 — upload to Kaggle, verify ~0.768 LB
6. **STOP HERE if val acc is bad. Debug before continuing.**
7. **Cells 14-17:** Train + evaluate seeds 123, 456
8. **Cell 18:** Ensemble comparison
9. **Cell 19:** Final ensemble submission

### Estimated Time (A100):
- Sanity check: ~2 min
- Full training (1 seed): ~15-25 min
- Val evaluation: ~5 min
- Test inference: ~5 min
- Total for 3 seeds + ensemble: ~1.5-2 hours